In [6]:
# CS-340 Project Two - Grazioso Salvare Dashboard
# Using my CRUD_Python_Module to connect to the AAC MongoDB
# This dashboard filters and visualizes animal data for search and rescue training

from jupyter_dash import JupyterDash
JupyterDash.infer_jupyter_proxy_config()

import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output

import base64, os
import pandas as pd
from CRUD_Python_Module import AnimalShelter

# ------------------------------------------------------------------
# Connection setup
# ------------------------------------------------------------------
# Switching between local and SNHU credentials depending on where I’m running it
use_local = True  # set to False before submission

if use_local:
    username, password = "aacuser", "Bryan0527"
    auth_db = "admin"
else:
    username, password = "aacuser", "SNHU1234"
    auth_db = "admin"

# connect to MongoDB using my CRUD module
db = AnimalShelter(username, password, auth_db=auth_db)

# helper function to pull Mongo data into a pandas DataFrame
def mongo_to_df(query: dict) -> pd.DataFrame:
    docs = db.read(query if isinstance(query, dict) else {})
    df = pd.DataFrame.from_records(list(docs))
    if "_id" in df.columns:
        df.drop(columns=["_id"], inplace=True)
    return df

# start with all data unfiltered
df = mongo_to_df({})

# ------------------------------------------------------------------
# Query builder for filter options
# ------------------------------------------------------------------
# These filters match what Grazioso Salvare looks for in dogs
def build_filter_query(filter_value: str) -> dict:
    if filter_value == "water":
        breeds = ["Labrador Retriever", "Chesapeake Bay Retriever", "Newfoundland"]
        return {
            "breed": {"$in": breeds},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    if filter_value == "mountain":
        breeds = ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]
        return {
            "breed": {"$in": breeds},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    if filter_value == "disaster":
        breeds = ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]
        return {
            "breed": {"$in": breeds},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
    return {}  # default to all records

# ------------------------------------------------------------------
# Dashboard layout (UI)
# ------------------------------------------------------------------
app = JupyterDash(__name__)

# load the logo image if it exists
logo_path = "GraziosoSalvareLogo.png"
if os.path.exists(logo_path):
    encoded = base64.b64encode(open(logo_path, "rb").read()).decode()
    logo_img = html.Img(src=f"data:image/png;base64,{encoded}", style={"height": "70px"})
else:
    logo_img = html.Div("Grazioso Salvare", style={"fontWeight": "600", "fontSize": "22px"})

# header with logo and my name
header_bar = html.Div(
    [
        logo_img,
        html.Div("CS-340 Dashboard • Emily Murphy", style={"fontSize": "18px", "marginTop": "10px"})
    ],
    style={"display": "flex", "gap": "16px", "alignItems": "center"}
)

# dashboard layout
app.layout = html.Div(
    [
        header_bar,
        html.Hr(),

        # radio buttons for filtering by rescue type
        html.Div(
            [
                html.Label("Rescue Type Filter", style={"fontWeight": "600"}),
                dcc.RadioItems(
                    id="filter-type",
                    options=[
                        {"label": "All", "value": "all"},
                        {"label": "Water Rescue", "value": "water"},
                        {"label": "Mountain or Wilderness", "value": "mountain"},
                        {"label": "Disaster or Individual Tracking", "value": "disaster"},
                    ],
                    value="all",
                    labelStyle={"display": "inline-block", "marginRight": "16px"}
                ),
            ],
            style={"marginBottom": "8px"}
        ),

        html.Hr(),

        # main data table
        dash_table.DataTable(
            id="datatable-id",
            columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
            data=df.to_dict("records"),
            page_size=10,
            filter_action="native",
            sort_action="native",
            sort_mode="multi",
            row_selectable="single",
            selected_rows=[],
            style_table={"overflowX": "auto"},
            style_cell={"fontSize": 12, "textAlign": "left"},
        ),

        html.Br(),
        html.Hr(),

        # chart and map side-by-side
        html.Div(
            className="row",
            style={"display": "flex", "gap": "16px"},
            children=[
                html.Div(id="graph-id", className="col s12 m6", style={"flex": 1}),
                html.Div(id="map-id", className="col s12 m6", style={"flex": 1}),
            ],
        ),
    ],
    style={"padding": "16px"}
)

# ------------------------------------------------------------------
# Callbacks (interactivity)
# ------------------------------------------------------------------
# updates the data table when I select a filter
@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")]
)
def update_table(filter_type):
    q = build_filter_query(filter_type if filter_type != "all" else "")
    dff = mongo_to_df(q)
    return dff.to_dict("records")

# updates the pie chart whenever the table changes
@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_chart(view_data):
    if not view_data:
        return html.Div("No data to chart.")
    dff = pd.DataFrame(view_data)
    names_col = "breed" if "breed" in dff.columns else dff.columns[0]
    fig = px.pie(dff, names=names_col, title="Breed Distribution")
    return dcc.Graph(figure=fig)

# highlights any selected columns in the table
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def highlight_cols(selected_columns):
    return [{"if": {"column_id": i}, "background_color": "#D2F3FF"} for i in (selected_columns or [])]

# updates the map when I select a row
@app.callback(
    Output("map-id", "children"),
    [Input("datatable-id", "derived_virtual_data"),
     Input("datatable-id", "derived_virtual_selected_rows")]
)
def update_map(view_data, selected_rows):
    if not view_data:
        return html.Div("No rows to map.")
    dff = pd.DataFrame(view_data)
    row = 0 if not selected_rows else selected_rows[0]

    # try to find the correct column names for coordinates
    lat_candidates = [c for c in dff.columns if c.lower() in ("location_lat", "latitude", "lat")]
    lon_candidates = [c for c in dff.columns if c.lower() in ("location_long", "longitude", "long", "lng")]
    name_candidates = [c for c in dff.columns if c.lower() in ("name", "animal_name")]
    breed_candidates = [c for c in dff.columns if c.lower() == "breed"]

    lat_col = lat_candidates[0] if lat_candidates else dff.columns[13]
    lon_col = lon_candidates[0] if lon_candidates else dff.columns[14]
    name_col = name_candidates[0] if name_candidates else dff.columns[9]
    breed_col = breed_candidates[0] if breed_candidates else dff.columns[4]

    # set fallback location if missing coords
    try:
        lat = float(dff.iloc[row][lat_col])
        lon = float(dff.iloc[row][lon_col])
    except Exception:
        lat, lon = 30.75, -97.48

    return [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(str(dff.iloc[row][breed_col])),
                        dl.Popup([html.H1("Animal Name"), html.P(str(dff.iloc[row][name_col]))]),
                    ],
                ),
            ],
        )
    ]

# ------------------------------------------------------------------
# Run the dashboard
# ------------------------------------------------------------------
app.run_server(mode="inline", host="0.0.0.0", port=8060, debug=False)
